## 1. 📦 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
import xgboost as xgb

np.random.seed(42)
print("✅ Libraries loaded successfully")


## 2. 📥 Load Dataset


In [ ]:
import os

DATASET_PATH = None
for candidate in [
    "/root/.cache/kagglehub/datasets/adrianmcmahon/imdb-india-movies/versions/1/IMDb Movies India.csv",
    "IMDb Movies India.csv",
]:
    if os.path.exists(candidate):
        DATASET_PATH = candidate
        break

if DATASET_PATH:
    df_raw = pd.read_csv(DATASET_PATH, encoding="latin-1")
    print(f"✅ Loaded real dataset: {DATASET_PATH}")
else:
    print("⚠️  Real dataset not found — generating realistic simulation...")

    n = 2000
    genres_pool = ["Drama","Action","Comedy","Romance","Thriller","Crime",
                   "Horror","Biography","Musical","Adventure","Family",
                   "Mystery","Fantasy","Animation","Documentary"]
    directors = [
        "Rajkumar Hirani","Zoya Akhtar","Anurag Kashyap","Mani Ratnam",
        "Farah Khan","Vishal Bhardwaj","Sanjay Leela Bhansali","Shoojit Sircar",
        "Imtiaz Ali","Dibakar Banerjee","Ram Gopal Varma","Priyadarshan",
        "Rohit Shetty","Kabir Khan","Nitesh Tiwari","Tigmanshu Dhulia",
        "Neeraj Pandey","Meghna Gulzar","Hansal Mehta","Onir",
    ]
    actors_pool = [
        "Amitabh Bachchan","Shah Rukh Khan","Aamir Khan","Salman Khan",
        "Hrithik Roshan","Ranbir Kapoor","Ranveer Singh","Akshay Kumar",
        "Deepika Padukone","Priyanka Chopra","Alia Bhatt","Kareena Kapoor",
        "Vidya Balan","Tabu","Konkona Sen Sharma","Nawazuddin Siddiqui",
        "Irrfan Khan","Manoj Bajpayee","Pankaj Tripathi","Ayushmann Khurrana",
    ]

    director_quality = {d: np.random.uniform(0.3, 1.0) for d in directors}
    director_quality.update({
        "Rajkumar Hirani":0.95,"Zoya Akhtar":0.90,"Anurag Kashyap":0.88,
        "Mani Ratnam":0.92,"Vishal Bhardwaj":0.87,"Shoojit Sircar":0.89,
        "Nitesh Tiwari":0.85,"Meghna Gulzar":0.84,"Neeraj Pandey":0.82,
    })
    genre_effect = {
        "Drama":0.15,"Biography":0.20,"Crime":0.10,"Thriller":0.08,
        "Musical":0.05,"Comedy":-0.05,"Action":-0.10,"Horror":-0.15,
        "Romance":0.00,"Adventure":0.02,"Family":0.03,"Mystery":0.12,
        "Fantasy":-0.02,"Animation":0.05,"Documentary":0.18,
    }

    records = []
    for i in range(n):
        year      = np.random.randint(1950, 2020)
        director  = np.random.choice(directors)
        genre_list= np.random.choice(genres_pool, size=np.random.randint(1,4), replace=False)
        actor_list= np.random.choice(actors_pool, size=np.random.randint(1,4), replace=False)
        votes     = int(np.random.lognormal(7.5, 1.8))
        duration  = np.random.randint(80, 200)
        rating    = (5.5
                     + 2.5*director_quality[director]
                     + np.mean([genre_effect.get(g,0) for g in genre_list])
                     + 0.3*np.log1p(votes)/np.log1p(100000)
                     - 0.002*abs(duration-140)
                     + 0.005*(year-1980)/40
                     + np.random.normal(0, 0.5))
        rating = round(np.clip(rating, 1.0, 10.0), 1)
        records.append({
            "Name":     f"Movie_{i+1}",
            "Year":     year,
            "Duration": f"{duration} min",
            "Genre":    ", ".join(genre_list),
            "Rating":   rating,
            "Votes":    votes,
            "Director": director,
            "Actor 1":  actor_list[0],
            "Actor 2":  actor_list[1] if len(actor_list)>1 else np.nan,
            "Actor 3":  actor_list[2] if len(actor_list)>2 else np.nan,
        })
    df_raw = pd.DataFrame(records)
    print(f"✅ Simulated {len(df_raw)} movies")

print(f"\nShape: {df_raw.shape}")
df_raw.head()


## 3. 🔍 Exploratory Data Analysis

In [ ]:
print("=== Data Types ===")
print(df_raw.dtypes)
print("\n=== Missing Values ===")
print(df_raw.isnull().sum())


In [ ]:
print("=== Rating Statistics ===")
df_raw["Rating"].describe()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
plt.style.use("seaborn-v0_8-darkgrid")

# Rating distribution
axes[0].hist(df_raw["Rating"], bins=30, color="#e8b86d", edgecolor="white", alpha=0.9)
axes[0].axvline(df_raw["Rating"].mean(), color="#7ec8e3", lw=2, ls="--",
                label=f"Mean {df_raw['Rating'].mean():.2f}")
axes[0].set_title("Rating Distribution", fontsize=13)
axes[0].set_xlabel("Rating"); axes[0].set_ylabel("Count")
axes[0].legend()

# Votes vs Rating
axes[1].scatter(np.log1p(df_raw["Votes"]), df_raw["Rating"],
                alpha=0.4, s=12, color="#c77dff")
axes[1].set_title("Log(Votes) vs Rating", fontsize=13)
axes[1].set_xlabel("Log(Votes+1)"); axes[1].set_ylabel("Rating")

# Ratings over Years
yr_grp = df_raw.groupby("Year")["Rating"].mean()
axes[2].plot(yr_grp.index, yr_grp.values, color="#56cfaa", lw=1.5)
axes[2].set_title("Avg Rating over Years", fontsize=13)
axes[2].set_xlabel("Year"); axes[2].set_ylabel("Avg Rating")

plt.tight_layout()
plt.show()


## 4. 🔧 Data Preprocessing

In [ ]:
df = df_raw.copy()

# Clean Year
df["Year"] = df["Year"].astype(str).str.extract(r"(\d{4})").astype(float)

# Clean Duration → numeric minutes
df["Duration_min"] = df["Duration"].astype(str).str.extract(r"(\d+)").astype(float)

# Clean Votes
if df["Votes"].dtype == object:
    df["Votes"] = df["Votes"].astype(str).str.replace(",","").str.extract(r"(\d+)").astype(float)

# Drop rows with missing target
df = df.dropna(subset=["Rating"])
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df = df.dropna(subset=["Rating"])

# Impute numerics
df["Year"].fillna(df["Year"].median(), inplace=True)
df["Duration_min"].fillna(df["Duration_min"].median(), inplace=True)
df["Votes"].fillna(0, inplace=True)

print(f"✅ Clean dataset shape: {df.shape}")
df[["Year","Duration_min","Votes","Rating"]].describe()


## 5. ⚙️ Feature Engineering

In [ ]:
# Log votes (handles heavy skew)
df["Log_Votes"] = np.log1p(df["Votes"])

# Movie age
df["Movie_Age"] = 2024 - df["Year"]

# Genre features
df["Genre"] = df["Genre"].fillna("Unknown")
df["Genre_Count"]   = df["Genre"].apply(lambda x: len(str(x).split(",")))
df["Primary_Genre"] = df["Genre"].apply(lambda x: str(x).split(",")[0].strip())

# Director frequency
dir_freq = df["Director"].value_counts()
df["Director_Frequency"] = df["Director"].map(dir_freq).fillna(1)

# Director average rating (Bayesian smoothing)
global_mean = df["Rating"].mean()
smoothing   = 5
dir_stats   = df.groupby("Director")["Rating"].agg(["mean","count"])
dir_stats["smoothed"] = (
    dir_stats["mean"] * dir_stats["count"] + global_mean * smoothing
) / (dir_stats["count"] + smoothing)
df["Director_Avg_Rating"] = df["Director"].map(dir_stats["smoothed"]).fillna(global_mean)

# Actor popularity
all_actors = pd.concat([df["Actor 1"], df["Actor 2"], df["Actor 3"]]).dropna()
actor_freq = all_actors.value_counts()
for col in ["Actor 1", "Actor 2", "Actor 3"]:
    df[f"{col}_freq"] = df[col].map(actor_freq).fillna(0)
df["Actor_Popularity"] = (df["Actor 1_freq"] + df["Actor 2_freq"] + df["Actor 3_freq"]) / 3

# Label encode primary genre
le = LabelEncoder()
df["Primary_Genre_Enc"] = le.fit_transform(df["Primary_Genre"].astype(str))

FEATURES = [
    "Year","Duration_min","Log_Votes","Movie_Age","Genre_Count",
    "Director_Frequency","Director_Avg_Rating","Actor_Popularity","Primary_Genre_Enc"
]

print("✅ Features ready:", FEATURES)
df[FEATURES + ["Rating"]].head()


### 5.1 Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 7))
corr = df[FEATURES + ["Rating"]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Feature Correlation Matrix", fontsize=14, pad=12)
plt.tight_layout()
plt.show()


## 6. ✂️ Train / Test Split

In [ ]:
TARGET = "Rating"
X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train size : {len(X_train)}")
print(f"Test  size : {len(X_test)}")


## 7. 🤖 Model Training & Evaluation

In [ ]:
models = {
    "Ridge Regression":  Ridge(alpha=1.0),
    "Lasso Regression":  Lasso(alpha=0.01),
    "Random Forest":     RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42),
    "XGBoost":           xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42, verbosity=0),
}

results = {}
for name, model in models.items():
    use_scaled = name in ["Ridge Regression", "Lasso Regression"]
    Xtr = X_train_sc if use_scaled else X_train.values
    Xte = X_test_sc  if use_scaled else X_test.values

    model.fit(Xtr, y_train)
    preds = model.predict(Xte)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    r2   = r2_score(y_test, preds)
    cv   = -cross_val_score(model, Xtr, y_train, cv=5, scoring="neg_root_mean_squared_error").mean()

    results[name] = {"model": model, "preds": preds,
                     "RMSE": rmse, "MAE": mae, "R2": r2, "CV_RMSE": cv,
                     "scaled": use_scaled}
    print(f"  {name:<28}  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}  CV-RMSE={cv:.4f}")

best_name = min(results, key=lambda k: results[k]["RMSE"])
print(f"\n🏆 Best model: {best_name}  (RMSE={results[best_name]['RMSE']:.4f})")


### 7.1 Results Summary Table

In [ ]:
summary = pd.DataFrame({
    "Model":    list(results.keys()),
    "RMSE":     [results[m]["RMSE"]    for m in results],
    "MAE":      [results[m]["MAE"]     for m in results],
    "R²":       [results[m]["R2"]      for m in results],
    "CV-RMSE":  [results[m]["CV_RMSE"] for m in results],
}).sort_values("RMSE").reset_index(drop=True)

summary.style.background_gradient(subset=["RMSE","MAE","CV-RMSE"], cmap="RdYlGn_r") \
             .background_gradient(subset=["R²"], cmap="RdYlGn") \
             .format({"RMSE":"{:.4f}","MAE":"{:.4f}","R²":"{:.4f}","CV-RMSE":"{:.4f}"})


## 8. 📊 Visualisations

In [ ]:
best = results[best_name]
preds_best = best["preds"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Model comparison
model_names = list(results.keys())
rmses  = [results[m]["RMSE"] for m in model_names]
colors = ["#e8b86d" if m == best_name else "#7ec8e3" for m in model_names]
axes[0].bar(range(len(model_names)), rmses, color=colors, edgecolor="white", alpha=0.85)
axes[0].set_xticks(range(len(model_names)))
axes[0].set_xticklabels([m.replace(" ", "\n") for m in model_names], fontsize=8)
axes[0].set_ylabel("RMSE"); axes[0].set_title("Model RMSE Comparison", fontsize=13)
for i,v in enumerate(rmses):
    axes[0].text(i, v+0.003, f"{v:.3f}", ha="center", fontsize=8)

# Predicted vs Actual
axes[1].scatter(y_test, preds_best, alpha=0.45, s=15, color="#c77dff")
lo, hi = min(y_test.min(), preds_best.min()), max(y_test.max(), preds_best.max())
axes[1].plot([lo,hi],[lo,hi], "r--", lw=2, label="Perfect")
axes[1].set_xlabel("Actual Rating"); axes[1].set_ylabel("Predicted Rating")
axes[1].set_title(f"{best_name}\nPredicted vs Actual", fontsize=13)
axes[1].legend()
axes[1].text(0.05, 0.90, f"R²={best['R2']:.4f}\nRMSE={best['RMSE']:.4f}",
             transform=axes[1].transAxes, fontsize=9,
             bbox=dict(boxstyle="round", fc="lightyellow", ec="gray"))

# Residuals
residuals = y_test.values - preds_best
axes[2].scatter(preds_best, residuals, alpha=0.45, s=15, color="#56cfaa")
axes[2].axhline(0, color="red", lw=2, ls="--")
axes[2].set_xlabel("Predicted Rating"); axes[2].set_ylabel("Residual")
axes[2].set_title(f"{best_name} — Residuals", fontsize=13)

plt.tight_layout()
plt.show()


In [ ]:
# Feature Importance (best tree model)
tree_model_name = next((m for m in ["XGBoost","Gradient Boosting","Random Forest"] if m in results), None)

if tree_model_name:
    tm = results[tree_model_name]["model"]
    fi_df = pd.DataFrame({"Feature": FEATURES,
                           "Importance": tm.feature_importances_}).sort_values("Importance")
    colors_fi = ["#e8b86d" if v >= fi_df["Importance"].quantile(0.7) else "#7ec8e3"
                 for v in fi_df["Importance"]]
    plt.figure(figsize=(9, 5))
    plt.barh(fi_df["Feature"], fi_df["Importance"], color=colors_fi, edgecolor="white", alpha=0.9)
    plt.title(f"Feature Importance — {tree_model_name}", fontsize=13)
    plt.xlabel("Importance Score")
    plt.tight_layout()
    plt.show()


## 9. 🎯 Predict Rating for a New Movie

In [ ]:
def predict_movie(name, year, duration_min, votes, genre, director, actors):
    """Predict IMDB rating for a given movie."""
    genre_count   = len(genre.split(","))
    primary_genre = genre.split(",")[0].strip()

    dir_avg  = dir_stats["smoothed"].get(director, global_mean)
    dir_frq  = dir_freq.get(director, 1)
    act_pop  = np.mean([actor_freq.get(a, 0) for a in actors])

    try:
        genre_enc = le.transform([primary_genre])[0]
    except ValueError:
        genre_enc = 0   # unseen genre → 0

    row = pd.DataFrame([{
        "Year":               year,
        "Duration_min":       duration_min,
        "Log_Votes":          np.log1p(votes),
        "Movie_Age":          2024 - year,
        "Genre_Count":        genre_count,
        "Director_Frequency": dir_frq,
        "Director_Avg_Rating":dir_avg,
        "Actor_Popularity":   act_pop,
        "Primary_Genre_Enc":  genre_enc,
    }])

    best_model = results[best_name]["model"]
    X_inp = scaler.transform(row) if results[best_name]["scaled"] else row.values
    pred  = best_model.predict(X_inp)[0]

    print(f"🎬  Movie       : {name}")
    print(f"📅  Year        : {year}")
    print(f"🎭  Director    : {director}")
    print(f"🎬  Genre       : {genre}")
    print(f"⭐  Predicted Rating : {pred:.2f} / 10")
    return pred

# Example prediction
predict_movie(
    name         = "Dil Dhadakne Do 2",
    year         = 2024,
    duration_min = 145,
    votes        = 50000,
    genre        = "Drama, Comedy",
    director     = "Zoya Akhtar",
    actors       = ["Ranveer Singh", "Priyanka Chopra", "Alia Bhatt"],
)
